# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import platform

# Install libomp-dev if on Linux (e.g., Colab environment)
if platform.system() == 'Linux':
    !apt-get update -qq
    !apt-get install -y libomp-dev -qq


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
if ... not in pdf.columns:
    pdf[...] = range(len(pdf))  # TODO: replace with your own identifier logic if provided
display(pdf.head())
# TODO: create a manageable subset (e.g., first 1000 rows)
pdf_subset = ...
pdf_subset[['id', 'title']].head()


In [ ]:
pdf = pd.read_csv(data_path, sep=';')

# Ensure an 'id' column exists. If not, create one.
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

# Create a manageable subset (e.g., first 1000 rows)
pdf_subset = pdf.head(1000).copy() # Use .copy() to avoid SettingWithCopyWarning
display(pdf_subset.head())

# Display 'id' and 'title' for verification
display(pdf_subset[['id', 'title']].head())


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Todo: create training examples from the subset data using the example_create_fn
faiss_train_examples = [...]
faiss_train_examples[:2]


In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Create training examples from the subset data using the example_create_fn
# Note: The original prompt mentioned InputExample(texts=[doc1]), but the provided function
# `example_create_fn(idx, text)` is more aligned with a row-wise application if `text` is the title.
# Assuming `faiss_train_examples` should be based on `pdf_subset['title']`.
faiss_train_examples = [example_create_fn(row['id'], row['title']) for idx, row in pdf_subset.iterrows()]

# Display the first two examples to verify
faiss_train_examples[:2]


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # TODO: encode the query using the sentence transformer model
    query_vector = model.encode(...)
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(...)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

display(search_content('animal', pdf_to_index, k=5))


In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # Encode the query using the sentence transformer model
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)

    # Perform the search in the FAISS index
    # The k argument specifies the number of nearest neighbors to retrieve
    sims, ids = index_content.search(query_vector, k)

    # Filter the DataFrame to get the results and add similarities
    # Since ids[0] is a numpy array of int64, directly use it for `isin`
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()

    # Map similarities to the results based on id order from search
    # Create a mapping from id to similarity
    sim_map = dict(zip(ids[0], sims[0]))
    results['similarities'] = results['id'].map(sim_map)

    # Sort results by similarity in descending order
    results = results.sort_values(by='similarities', ascending=False).reset_index(drop=True)
    return results

display(search_content('animal', pdf_to_index, k=5))


## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)
# To-Do: create the collection and add documents
collection = chroma_client.create_collection(...)
collection.add(...)
results = collection.query(...)
print(json.dumps(results, indent=2))


In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

# If a collection with the same name exists, delete it as per instruction
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

# Create the collection
collection = chroma_client.create_collection(name=collection_name)

# Add the first 100 titles of pdf_subset to the collection with their metadata and IDs
# Each document needs a content (text), metadata (dict), and id (string)
documents = pdf_subset['title'].head(100).tolist()
metadatas = pdf_subset[['topic']].head(100).to_dict(orient='records')
ids = [str(i) for i in pdf_subset['id'].head(100).tolist()]

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# Perform a query on the collection with the term "space"
# Retrieve the 10 most relevant documents
results = collection.query(
    query_texts=["space"],
    n_results=10
)
print(json.dumps(results, indent=2))


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA
# To-Do: create the text2text-generation pipeline
pipe = pipeline(...)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = pipe(prompt)[0]['generated_text']
print(response)


In [ ]:
model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA

# Create the text2text-generation pipeline
# Using AutoTokenizer and AutoModelForSeq2SeqLM (implicitly by pipeline for flan-t5-small)
pipe = pipeline("text2text-generation",
                model=model_id,
                tokenizer=model_id,
                max_new_tokens=512,
                device_map="auto") # Use device_map for potential GPU utilization

question = "What's the latest news on space development?"

# Ensure that results from ChromaDB query are available and in the expected format
# Assuming `results` from the previous cell is correctly populated.
# context_docs will be the list of documents from the ChromaDB query for 'space'.
# Taking the first query result's documents and limiting to top 3 as an example
context_docs = results['documents'][0][:3] # This assumes results['documents'] is a list of lists
context = ' '.join(context_docs)

# Construct the prompt as specified by the notebook, not the client context prompt example.
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

response = pipe(prompt)[0]['generated_text']
print(response)
